In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis with emcee: LBCO, HRPT

This tutorial demonstrates how to run Bayesian sampling with the
emcee minimizer and then resume the same chain from the saved project.

The workflow uses the same La0.5Ba0.5CoO3 powder diffraction example
as the DREAM Bayesian tutorial:

- run a short local refinement,
- derive finite fit bounds for the sampled parameters,
- switch to emcee and sample the posterior,
- save the project with the emcee chain,
- resume the chain with additional steps,
- inspect posterior plots after each sampling stage.

## Import Library

In [ ]:
import easydiffraction as ed

## Create a Project Container

The project is saved before sampling because emcee stores its chain in
the project's analysis sidecar file.

In [ ]:
project = ed.Project()

In [ ]:
project.save_as('projects/lbco_hrpt_emcee')

## Build the Structural Model

Define a compact cubic perovskite model for La0.5Ba0.5CoO3.

In [ ]:
project.structures.create(name='lbco')

In [ ]:
structure = project.structures['lbco']

In [ ]:
structure.space_group.name_h_m = 'P m -3 m'
structure.space_group.it_coordinate_system_code = '1'

In [ ]:
structure.cell.length_a = 3.88

In [ ]:
structure.atom_sites.create(
    label='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_type='Biso',
    adp_iso=0.2190,
)
structure.atom_sites.create(
    label='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_type='Biso',
    adp_iso=1.3916,
)

## Define the Diffraction Experiment

Download the HRPT powder pattern, create a neutron powder experiment,
and set the key instrument, peak-profile, and background values.

In [ ]:
data_path = ed.download_data(id=3, destination='data')

In [ ]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

In [ ]:
experiment = project.experiments['hrpt']

In [ ]:
experiment.linked_phases.create(id='lbco', scale=9.1351)

In [ ]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.0

In [ ]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1204
experiment.peak.broad_lorentz_y = 0.0844

In [ ]:
experiment.background.create(id='1', x=10, y=168.5585)
experiment.background.create(id='2', x=30, y=164.3357)
experiment.background.create(id='3', x=50, y=166.8881)
experiment.background.create(id='4', x=110, y=175.4006)

In [ ]:
experiment.excluded_regions.create(id='1', start=0, end=10)
experiment.excluded_regions.create(id='2', start=100, end=180)

## Run a Local Refinement First

The local fit provides starting values and uncertainties that are used
to build finite bounds for emcee.

In [ ]:
structure.cell.length_a.free = True

In [ ]:
experiment.linked_phases['lbco'].scale.free = True
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.instrument.calib_twotheta_offset.free = True

In [ ]:
project.analysis.minimizer.type = 'bumps (lm)'

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

In [ ]:
for param in project.free_parameters:
    param.set_fit_bounds_from_uncertainty()

In [ ]:
project.display.parameters.free()

## Run emcee Sampling

The sampling settings are intentionally small for tutorial runtime.
Use more steps and inspect convergence diagnostics for production
analysis.

In [ ]:
project.analysis.minimizer.type = 'emcee'

In [ ]:
project.analysis.minimizer.sampling_steps = 1000
project.analysis.minimizer.burn_in_steps = 200
project.analysis.minimizer.thinning_interval = 10
project.analysis.minimizer.population_size = 32
project.analysis.minimizer.initialization_method = 'ball'
project.analysis.minimizer.random_seed = 12345

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

## Inspect the Posterior

The posterior distribution plot shows the sampled marginal
distributions after the first emcee run.

In [ ]:
project.display.posterior.distribution()

The posterior predictive plot propagates the sampled parameter
uncertainty into the calculated diffraction pattern.

In [ ]:
project.display.posterior.predictive(expt_name='hrpt')

## Save the Sampled Project

Saving persists both the analysis state and the emcee chain sidecar so
the same chain can be resumed later.

In [ ]:
project.save()

## Resume emcee Sampling

Resume from the saved backend and append 500 more emcee steps to the
existing chain.

In [ ]:
project.analysis.fit(resume=True, extra_steps=500)

In [ ]:
project.display.fit.results()

## Inspect the Resumed Posterior

After resume, the posterior plots use the extended chain.

In [ ]:
project.display.posterior.distribution()

In [ ]:
project.display.posterior.predictive(expt_name='hrpt')